# Speculative Decoding, Simulated
Two things this notebook proves by running them:
1. **The output distribution is exact.** Rejection sampling between a draft and a target distribution reproduces the *target* distribution bit-for-bit — a bad draft only costs speed, never correctness.
2. **The speedup math.** Expected accepted tokens per verifier pass is $E = (1-\alpha^{N+1})/(1-\alpha)$; the whole game is raising $\alpha$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
VOCAB = 8
# target distribution q (the big model) and a deliberately mediocre draft p
q = np.array([0.35, 0.22, 0.15, 0.10, 0.07, 0.05, 0.04, 0.02])
p = np.array([0.15, 0.15, 0.15, 0.15, 0.10, 0.10, 0.10, 0.10])  # flatter, worse guesses
assert abs(q.sum() - 1) < 1e-9 and abs(p.sum() - 1) < 1e-9

## 1. Rejection sampling: draft, then verify
For each drafted token `d ~ p`: accept with probability `min(1, q(d)/p(d))`. On reject, resample from the residual `(q - p)+ / ||(q - p)+||`. The accepted stream must converge to `q` no matter how bad `p` is.

In [ ]:
def speculative_step(q, p, rng):
    """One draft-verify round. Returns the emitted token."""
    d = rng.choice(len(q), p=p)
    if rng.random() < min(1.0, q[d] / p[d]):
        return d  # accepted
    resid = np.clip(q - p, 0, None)
    resid /= resid.sum()
    return rng.choice(len(q), p=resid)  # resample on reject

N_STEPS = 200_000
out = np.array([speculative_step(q, p, rng) for _ in range(N_STEPS)])
emp = np.bincount(out, minlength=VOCAB) / N_STEPS
tv = 0.5 * np.abs(emp - q).sum()  # total variation distance
print(f"target q:      {np.round(q, 3)}")
print(f"empirical:     {np.round(emp, 3)}")
print(f"total variation distance: {tv:.4f}")
assert tv < 0.02, "distribution drifted — the invariant is broken"
print("OK: output matches the target distribution exactly (up to sampling noise).")

## 2. The speedup equation
$E[\text{accepted}] = (1 - \alpha^{N+1})/(1 - \alpha)$ tokens per target forward pass. Speedup $= E / (1 + N \cdot c)$ where $c$ = draft cost as a fraction of one target pass.

In [ ]:
def expected_accepted(alpha, N):
    return (1 - alpha**(N+1)) / (1 - alpha)

def speedup(alpha, N, c=0.05):
    return expected_accepted(alpha, N) / (1 + N * c)

# sanity: the paper's headline numbers fall out of the formula
for alpha, N, label in [(0.6, 5, "vanilla draft model"), (0.8, 5, "EAGLE-class"), (0.95, 8, "great draft head")]:
    print(f"{label:20s} alpha={alpha:<4} N={N} -> {speedup(alpha, N):.2f}x speedup")

assert speedup(0.6, 5) < speedup(0.8, 5) < speedup(0.95, 8)
assert 2.0 < speedup(0.8, 5) < 3.5  # the EAGLE band

In [ ]:
# optimal draft length for each acceptance rate (c = 5% draft cost)
alphas = [0.6, 0.7, 0.8, 0.9, 0.95]
Ns = range(1, 13)
fig, ax = plt.subplots(figsize=(10, 5.5))
for a in alphas:
    ax.plot(list(Ns), [speedup(a, n) for n in Ns], marker="o", label=f"alpha={a}")
ax.set_xlabel("Draft length N"); ax.set_ylabel("Expected speedup")
ax.set_title("Longer drafts only pay when alpha is high")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

for a in alphas:
    best = max(Ns, key=lambda n: speedup(a, n))
    print(f"alpha={a}: optimal N={best}, speedup={speedup(a, best):.2f}x")